In [ ]:
import csv
from pickle import EXT1

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
from matplotlib.collections import LineCollection
import matplotlib.collections as mcoll
plt.rcParams.update({
    "font.family": "serif",
    "axes.edgecolor": "#cfc8bb",
    "axes.labelcolor": "#444440",
    "xtick.color": "#8a8a84",
    "ytick.color": "#8a8a84",
    "text.color": "#1a1a18"
})

# VARIABLE ASSIGNMENT
class GutParameters:
    def __init__(self):
        # microbes
        self.Bp = 0.25*10**6 # initial population size of producer bacteria
        self.Bc = 0.12*10**6 # initial population size of consumer bacteria

        # nutrients
        self.NI = 0.451 # initial inulin content in the gut
        self.NFOS = 0.434 # initial FOS content in the gut

        # space and time constraints
        self.R = 3 # Replicates
        self.S = 5.9*10**9 # total available space
        self.Sp = 7.85 # size of producer bacteria
        self.Sc = 4.25 # size of consumer bacteria
        self.T = 240 # hours

        # kinetic constants
        self.Kp = 13.5 # Monod constant for Producer
        self.Kc = 6.98 # Monod constant for Consumer

        self.Rp = 0.23 # Growth Rate of Producer w unlimited food -> consider *10**8
        self.Rc = 0.37 # Growth Rate of Consumer w unlimited food -> consider *10**8

        self.Yp = 7.39*10**8 # Yield or Nutrient to Biomass Conversion Efficiency of Producer
        self.Yc = 4.30*10**8 # Yield or Nutrient to Biomass Conversion Efficiency of Consumer

        self.EC = 0.45 # Efficiency of Inulin to FOS Conversion by Producer
        self.EI = 0.95 # uptake efficiency of Inulin by Producer
        self.EFOS = 0.125 # uptake efficiency of FOS by Consumer

        # Fluid Mechanic Component inclusive of flow and diffusion
        self.taoB = 72 # Retention Time in hours
        self.taoN = 24 # Retention Time in hours

        # Meal Intake (Gaussian) Parameters
        self.meal_sigma = 0.5 # width of each standard deviation of the bell curve i.e. 3x0.5*2 => is 99.7% of the bell curve is covered in  3 hours
        self.inulin_MW = 0.504 # g/mmol  (MW ≈ 504 g/mol for short-chain inulin)
        self.V = 0.377 # L volume of the proximal colon
gp = GutParameters()

# import mess menu
meal_df_file_path = "/Users/vishikherotia/Downloads/AshokaMenu_CompBioHVS.csv" 

# import csv as dataframe
meal_df = pd.read_csv(meal_df_file_path) #find syntax

def get_meal_time(hour):
    # what is the meal to choose depending on the time of day and window on meal absorption
    if abs(hour - (8.0 + 1.5)) < 1.5: # input nutrients M introduced slowly in the span of 3 hours (1.5 on each side of the bell curve peak)
        return "breakfast" # 8am
    elif abs(hour - (13.0 + 1.5)) < 1.5:
        return "lunch" # 1pm
    elif abs(hour - (20.0 + 1.5)) < 1.5:
        return "dinner" #8pm
    else:
        return None

# MEAL TIME HELPER FX: PICK RANDOM 3 + 1 ITEMS FROM THE MENU
def meal_helper(meal_df):
    #list to store our meal_items
    meal_items = [] #list of dictionaries for each day and meal
    for day in range(1, 11):
        for meal_time in ['breakfast', 'lunch', 'dinner']:
            # create the search criteria as a "mask" + avoid capitalization differences by str.lower()
            meal_mask = (meal_df['day'] == day) & (meal_df['meal_time'].str.lower() == meal_time)
            # use this criteria to shortlist options to choose from for every day and every meal
            meal_options = meal_df[meal_mask]

            # SAFETY: confirm options =/= None
            if not meal_options.empty:
                # separate foods and drinks for selection afterwards
                food_items = meal_options[meal_options['item_type'].str.lower() == "food"]
                drink_items = meal_options[meal_options['item_type'].str.lower() == "drink"]

                # Randomly pick 3 food items + 1 drink item
                selected_food = random.sample(list(food_items['item_name']), min(len(food_items), 3)) #random.sample gives 3 unique values
                if not drink_items.empty:
                    selected_drink = random.choice(list(drink_items['item_name']))
                else:
                    selected_drink = None

                # find the fibre content in the food and drink items and sum them to return one singular value for each
                food_fibre = meal_options[meal_options['item_name'].isin(selected_food)]['dietary_fibre'].sum()
                drink_fibre = meal_options[meal_options['item_name'] == selected_drink]['dietary_fibre'].sum()

                meal_items.append({
                    "day": day,
                    "meal_time": meal_time,
                    "food_items": selected_food,
                    "drink_items": selected_drink,
                    "meal_dietary_fibre": food_fibre + drink_fibre
                })
    return pd.DataFrame(meal_items) #converts the list of dictionaries into a pandas dataframe

#run the meal_helper fx to create the randomized meal plan table (i.e. df) which we can use further to return just fibre values total per meal
meal_recruits_df = meal_helper(meal_df)

# MEAL TIME FX
def meal_recruiter(day, meal_time):
    # SAFETY: If it's not meal time, then fibre intake = zero !
        # NEW CHANGE --> only ask for meal_recruiter if it is meal_time from get_meal_time fx. so this def decreases time complexity by alot ! (good one, hitee proud :)))
    if meal_time is None:
        return 0.0
    # criteria for search i.e. what day and meal time it is
    meal_recruited_items_mask = (meal_recruits_df['day'] == day) & (meal_recruits_df['meal_time'].str.lower() == meal_time.lower())

    result = meal_recruits_df[meal_recruited_items_mask]

    if not result.empty:
        # it asks for only the first row result () from
        meal_dietary_fibre = result.iloc[0]['meal_dietary_fibre']

        return meal_dietary_fibre
    else:
        return 0.0


meal_recruits_df.to_csv("meal_recruits.csv", index=False)

# FUNCTIONS FOR CALCULATION OF SYSTEM STATES AND INTEGRATION OVER T = 240hours

# functions which calculate change in each state (P, C, I, FOS) of the system for 1dt
def producer(Bp, Bc, Rp, NI, Kp, Sp, Sc, S, taoB):
    d_Bp = Bp * Rp * (NI / (NI + Kp)) * (1 - (Bp * Sp + Bc * Sc) / S) - (1/taoB) * Bp
    return d_Bp

def consumer(Bc, Bp, Rc, NFOS, Kc, Sp, Sc, S, taoB):
    d_Bc = Bc * Rc * (NFOS / (NFOS + Kc)) * (1 - (Bp * Sp + Bc * Sc) / S) - (1/taoB) * Bc
    return d_Bc

def inulin(NI, Rp, Yp, Kp, Bp, taoN, inulin_MW, hour, day, meal_time):
    if meal_time is None:
        NIinflow_rate = 0.0
    else: # // *IMPORTANT* -> ANNOTATE THIS SECTION ONCE IT WORKS PLS PLS PLS
        fibre_g = meal_recruiter(day, meal_time)

        meal_hours = {"breakfast": 8.0, "lunch": 13.0, "dinner": 20.0}
        peak = meal_hours[meal_time] + 1.5

        deltaT = hour - peak # how much away you are in time from the peak fibre conc.

        gauss = (np.exp(-0.5 * (deltaT / gp.meal_sigma) ** 2) / (gp.meal_sigma * np.sqrt(2 * np.pi)))

        NIinflow_rate = ((fibre_g * 0.15 * gp.inulin_MW)/ gp.V) * gauss # 0.15 is fraction off dietary fibre inulin is
    d_NI = NIinflow_rate - (Rp / Yp) * (NI / (NI + Kp)) * Bp - (1/taoN) * NI
    return d_NI

def fos(NFOS, NI, EC, EI, EFOS, Rc, Yc, Kc, Bc, Bp, taoN, Rp, Yp, Kp):
    d_NFOS = EC * EI * (Rp / Yp) * (NI/ (NI + Kp)) * Bp - (Rc / Yc) * EFOS * (NFOS / (NFOS + Kc)) * Bc - (1/taoN) * NFOS
    return d_NFOS

# function that updates the state of the system at each dt
def update(system_state, gp, hour, day, meal_time):
    Bp, Bc, NI, NFOS = system_state
    # update functions
    d_Bp = producer(Bp, Bc, gp.Rp, NI, gp.Kp, gp.Sp, gp.Sc, gp.S, gp.taoB)
    d_Bc = consumer(Bc, Bp, gp.Rc, NFOS, gp.Kc, gp.Sp, gp.Sc, gp.S, gp.taoB)
    d_NI = inulin(NI, gp.Rp, gp.Yp, gp.Kp, Bp, gp.taoN, gp.inulin_MW, hour, day, meal_time)
    d_NFOS = fos(NFOS, NI, gp.EC, gp.EI, gp.EFOS, gp.Rc, gp.Yc, gp.Kc, Bc, Bp, gp.taoN, gp.Rp, gp.Yp, gp.Kp)

    return np.array([d_Bp, d_Bc, d_NI, d_NFOS])

# integration method (RK4)
def rk4(sys_state, gp, dt, hour, day, meal_time):
    # slope at the start of the interval
   k1 = update(sys_state, gp, hour, day, meal_time) * dt
    # slope at midpoint from k1
   k2 = update(sys_state + k1/2, gp, hour, day, meal_time) * dt
    # slope at midpoint again from k2
   k3 = update(sys_state + k2/2, gp, hour, day, meal_time) * dt
    # slope at the end of interval from k3
   k4 = update(sys_state + k3, gp, hour, day, meal_time) * dt

   return sys_state + (k1 + 2*k2 + 2*k3 + k4)/6

# RUN SIMULATION:
dt = 0.001 # as less as possible for rk4
steps = 240000
time = (steps+1)*dt # total time should be = 240 hours
T_list = np.arange(0, time, dt) # (start, stop, step)

# keep space for storing system states as an empty numpy array
system_states = np.empty((steps+1, 4))
system_states[0] = np.array([gp.Bp, gp.Bc, gp.NI, gp.NFOS]) #header row

# INTEGRATION LOOP (each step)
for i in range(steps):
    # calculate total time passed in hours (:now)
    current_time = i * dt

    # infer 1. time of the day i.e. hour 2. day from 1:11
    hour = current_time % 24
    day = int(current_time // 24) + 1 # quotient without decimal i.e. how many 24s have passed i.e. days.
    # +1 bcoz it starts from day 1 not 0
    meal_time = get_meal_time(hour)

    # integrate for all system_states step by step
    system_states[i+1] = rk4(system_states[i], gp, dt, hour, day, meal_time)

# extract each column as a list to plot
Bp = system_states[:, 0]
Bc = system_states[:, 1]
NI = system_states[:, 2]
NFOS = system_states[:, 3]


BG_COLOR = "#f7f4ee"
MOSS = "#2d5a3d"
AMBER = "#8b6914"
INK = "#8a8a84"

def finalize_plot(fig, ax, filename): #plot styling
    
    ax.set_facecolor(BG_COLOR)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color("#cfc8bb")
    ax.spines['bottom'].set_color("#cfc8bb")
    ax.grid(alpha=0.1)
    fig.patch.set_facecolor(BG_COLOR)
    plt.tight_layout()
    plt.savefig(f"{filename}.png", dpi=300, bbox_inches='tight')
    plt.show()


# PLOT 1: Nutrient Timeseries

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(T_list, NI, color=MOSS, lw=2, label="Inulin")
ax1.plot(T_list, NFOS, color=AMBER, lw=2, label="FOS")
ax1.set_xlim(0, 240) 

# Add ticks every 24 hours to mark the days
ax1.set_xticks(np.arange(0, 241, 24))
ax1.set_xticklabels([f"D{d}" for d in range(11)])
ax1.set_title("Nutrient Concentrations Over Time", fontsize=14, pad=15)
ax1.set_xlabel("Time (hours)")
ax1.set_ylabel("Concentration (mM)")
ax1.legend(frameon=False)
finalize_plot(fig1, ax1, "1_nutrient_timeseries")


# PLOT 2: Bacterial Timeseries

fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.plot(T_list, Bp, color=MOSS, lw=2, label="B. theta")
ax2.plot(T_list, Bc, color=AMBER, lw=2, label="B. breve")
ax2.set_xlim(0, 240) 

# Add ticks every 24 hours to mark the days
ax2.set_xticks(np.arange(0, 241, 24))
ax2.set_xticklabels([f"D{d}" for d in range(11)])
ax2.set_title("Bacterial Population Dynamics", fontsize=14, pad=15)
ax2.set_xlabel("Time (hours)")
ax2.set_ylabel("Population (CFU/ml)")
ax2.legend(frameon=False)
finalize_plot(fig2, ax2, "2_bacteria_timeseries")


# PLOT 3: Nutrient Phase Portrait (With Time Gradient)

fig3, ax3 = plt.subplots(figsize=(7, 6))
pts_n = np.array([NI, NFOS]).T.reshape(-1, 1, 2)
seg_n = np.concatenate([pts_n[:-1], pts_n[1:]], axis=1)
lc_n = mcoll.LineCollection(seg_n, cmap='YlGn', lw=1.5, alpha=0.8)
lc_n.set_array(np.linspace(0, 1, len(NI)))
ax3.add_collection(lc_n)

# Markers (Start/End)
ax3.scatter(NI[0], NFOS[0], color="#5a9e6e", s=120, edgecolors='white', zorder=10, label="Start")
ax3.scatter(NI[-1], NFOS[-1], color=MOSS, marker='s', s=120, edgecolors='white', zorder=10, label="End")

ax3.set_xlim(min(NI)*0.9, max(NI)*1.1)
ax3.set_ylim(min(NFOS)*0.9, max(NFOS)*1.1)
ax3.set_title("Phase Portrait: FOS vs Inulin", fontsize=13)
ax3.set_xlabel("Inulin (mM)")
ax3.set_ylabel("FOS (mM)")
fig3.colorbar(lc_n, ax=ax3, label="Time Progression →")
finalize_plot(fig3, ax3, "3_nutrient_phase")


# PLOT 4: Bacterial Phase Portrait (With Time Gradient)
# ─────────────────────────────────────────────────────────────────────────────
fig4, ax4 = plt.subplots(figsize=(7, 6))
pts_b = np.array([Bp, Bc]).T.reshape(-1, 1, 2)
seg_b = np.concatenate([pts_b[:-1], pts_b[1:]], axis=1)
lc_b = mcoll.LineCollection(seg_b, cmap='YlOrBr', lw=1.5, alpha=0.8)
lc_b.set_array(np.linspace(0, 1, len(Bp)))
ax4.add_collection(lc_b)

# Markers (Start/End)
ax4.scatter(Bp[0], Bc[0], color="#5a9e6e", s=120, edgecolors='white', zorder=10)
ax4.scatter(Bp[-1], Bc[-1], color=AMBER, marker='s', s=120, edgecolors='white', zorder=10)

ax4.set_xlim(min(Bp)*0.9, max(Bp)*1.1)
ax4.set_ylim(min(Bc)*0.9, max(Bc)*1.1)
ax4.set_title("Phase Portrait: Bacterial Co-dynamics", fontsize=13)
ax4.set_xlabel("B. theta (CFU/ml)")
ax4.set_ylabel("B. breve (CFU/ml)")
cbar = fig4.colorbar(lc_b, ax=ax4)
cbar.set_label("Time progression →", color=INK)
finalize_plot(fig4, ax4, "4_bacteria_phase")